In [22]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

In [23]:
ACS_CSV = "acs_la_tract_2022.csv"
TRACTS_GEO = "la_tracts.geojson"
CRIME_CSV = "data/Crime_Data_from_2020_to_Present.csv"

In [24]:
acs = pd.read_csv(ACS_CSV, dtype={"GEOID": str})
tracts = gpd.read_file(TRACTS_GEO)
crime = pd.read_csv(CRIME_CSV)
acs.head(2), tracts.head(2), crime.head(2)

(        geoid  total_pop  median_income  state  county   tract  \
 0  6037101110       4014        68972.0      6      37  101110   
 1  6037101122       4164       118859.0      6      37  101122   
 
                              name  b01003_001_moe  b19013_001_moe  
 0  Los Angeles County, California           473.0         17023.0  
 1  Los Angeles County, California           822.0         31445.0  ,
          GEOID  B01003_001E state county   tract  \
 0  06037670413         4852    06    037  670413   
 1  06037650101         5532    06    037  650101   
 
                              NAME  B01003_001_moe  \
 0  Los Angeles County, California           540.0   
 1  Los Angeles County, California           906.0   
 
                                             geometry  
 0  POLYGON ((-118.4037 33.78136, -118.39236 33.78...  
 1  POLYGON ((-118.32645 33.8728, -118.31779 33.87...  ,
        DR_NO               Date Rptd                DATE OCC  TIME OCC  AREA  \
 0  211507896 

In [25]:
def to_snake(s): 
    return s.strip().lower().replace(" ", "_").replace("-", "_")

def standardize(df): 
    df = df.copy(); df.columns = [to_snake(c) for c in df.columns]; return df

In [26]:
acs = standardize(acs).rename(columns={"b01003_001e":"total_pop","b19013_001e":"median_income"})
tracts = standardize(tracts)
crime = standardize(crime)

In [27]:
# ensure GEOID present
if "geoid" not in tracts.columns:
    for cand in ["geoid10","GEOID","tractce","TRACTCE"]:
        if cand.lower() in tracts.columns:
            tracts = tracts.rename(columns={cand.lower():"geoid"}); break


In [28]:
# minimal crime cleaning (fast)
crime = crime.rename(columns={"area_name":"area_name"})  # no-op; keep if exists
needed = ["date_occ","crm_cd_desc","vict_age","lat","lon"]
for c in needed:
    if c not in crime.columns:
        pass  # continue; this notebook focuses on spatial join

In [29]:
# keep only valid coords
crime = crime.dropna(subset=["lat","lon"])
crime = crime[(crime["lat"].between(-90,90)) & (crime["lon"].between(-180,180))]

In [30]:
# completeness snapshot (for rubric)
display(acs[["geoid","total_pop","median_income"]].isnull().sum())


geoid             0
total_pop         0
median_income    49
dtype: int64

In [31]:
print("ACS GEOID dtype:", acs["geoid"].dtype)
print("Tracts GEOID dtype:", tracts["geoid"].dtype)

print("\nSample ACS GEOIDs:", acs["geoid"].head().tolist())
print("Sample Tract GEOIDs:", tracts["geoid"].head().tolist())

ACS GEOID dtype: int64
Tracts GEOID dtype: object

Sample ACS GEOIDs: [6037101110, 6037101122, 6037101220, 6037101221, 6037101222]
Sample Tract GEOIDs: ['06037670413', '06037650101', '06037620303', '06037574000', '06037571400']


In [32]:
acs["geoid"] = acs["geoid"].astype(str).str.zfill(11)
tracts["geoid"] = tracts["geoid"].astype(str).str.zfill(11)

In [33]:
tracts_acs = tracts.merge(
    acs[["geoid", "total_pop", "median_income"]],
    on="geoid", how="left"
)

tracts_acs

,geoid,b01003_001e,state,county,tract,name,b01003_001_moe,geometry,total_pop,median_income
0,06037670413,4852,06,037,670413,"Los Angeles County, California",540.0,"POLYGON ((-118.4037 33.78136, -118.39236 33.78...",4852,182212.0
1,06037650101,5532,06,037,650101,"Los Angeles County, California",906.0,"POLYGON ((-118.32645 33.8728, -118.31779 33.87...",5532,105417.0
2,06037620303,4934,06,037,620303,"Los Angeles County, California",597.0,"POLYGON ((-118.41 33.89055, -118.40791 33.8924...",4934,202454.0
3,06037574000,5178,06,037,574000,"Los Angeles County, California",461.0,"POLYGON ((-118.11344 33.81412, -118.11308 33.8...",5178,141536.0
4,06037571400,5018,06,037,571400,"Los Angeles County, California",522.0,"POLYGON ((-118.16432 33.84706, -118.16346 33.8...",5018,125370.0
...,...,...,...,...,...,...,...,...,...,...
2490,06037401901,3823,06,037,401901,"Los Angeles County, California",155.0,"POLYGON ((-117.71668 34.10064, -117.71491 34.1...",3823,87083.0
2491,06037111302,4880,06,037,111302,"Los Angeles County, California",647.0,"POLYGON ((-118.52332 34.25734, -118.52215 34.2...",4880,88458.0
2492,06037115103,2725,06,037,115103,"Los Angeles County, California",67.0,"POLYGON ((-118.53392 34.24456, -118.5274 34.24...",2725,NaN
2493,06037241202,5291,06,037,241202,"Los Angeles County, California",915.0,"POLYGON ((-118.29164 33.9382, -118.28694 33.93...",5291,51328.0
